# DIMER Language-Model Adapter — Artifact Inference Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb)

This notebook consumes an adapter artifact produced by the standalone fine-tuning tutorial.

Flow:

> upload ZIP → safe extraction → verify manifest and SHA-256 hashes → read immutable base-model
> provenance → load exact base revision → attach PEFT adapter → generate

The original training dataset is **not required** and is not expected inside the artifact.

**AI provenance:** OpenAI / ChatGPT — GPT-5.6 Sol High; Builder role. This is implementation
provenance, not independent reviewer sign-off.

## 1. Install the inference runtime

In [ ]:
%pip -q install \
  transformers==4.57.1 \
  tokenizers==0.22.1 \
  huggingface-hub==0.36.0 \
  peft==0.18.0 \
  accelerate==1.11.0 \
  bitsandbytes==0.49.0 \
  safetensors==0.8.0

In [ ]:
import hashlib
import json
import platform
import re
import shutil
import stat
import zipfile
from pathlib import Path

import pandas as pd
import torch
import peft
import transformers

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required. "
        "In Colab: Runtime → Change runtime type → GPU."
    )
print("GPU:", torch.cuda.get_device_name(0))

AI_PROVENANCE = {
    "provider": "OpenAI",
    "product": "ChatGPT",
    "model": "GPT-5.6 Sol High",
    "role": "Builder",
    "note": "Implementation provenance; not independent reviewer sign-off.",
}


## 2. Upload and verify the artifact

Optionally paste the whole-ZIP SHA-256 printed by the training notebook. The manifest inside
the ZIP is always verified even when this optional outer digest is blank.

In [ ]:
EXPECTED_ARTIFACT_ZIP_SHA256 = ""  # @param {type:"string"}

from google.colab import files

uploaded = files.upload()
zip_names = [
    name
    for name in uploaded
    if name.lower().endswith(".zip")
]
if len(uploaded) != 1 or len(zip_names) != 1:
    raise ValueError(
        "Upload exactly one artifact ZIP."
    )

zip_path = (
    Path("/content")
    / Path(zip_names[0]).name
)
zip_path.write_bytes(
    uploaded[zip_names[0]]
)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(
            lambda: fh.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


actual_zip_sha = sha256_file(zip_path)
print(
    "Artifact ZIP SHA-256:",
    actual_zip_sha,
)
if EXPECTED_ARTIFACT_ZIP_SHA256:
    if (
        actual_zip_sha.lower()
        != EXPECTED_ARTIFACT_ZIP_SHA256
        .strip()
        .lower()
    ):
        raise ValueError(
            "Whole-ZIP SHA-256 does not "
            "match the expected value."
        )

In [ ]:
MAX_ARTIFACT_BYTES = (
    512 * 1024 * 1024
)
extract_root = Path(
    "/content/dimer-language-model-artifact"
).resolve()

if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)


def manifest_member_path(root, member):
    root = Path(root).resolve()
    member_path = Path(member)
    if (
        member_path.is_absolute()
        or ".." in member_path.parts
    ):
        raise ValueError(
            f"Unsafe artifact path: {member!r}"
        )

    resolved = (
        root / member_path
    ).resolve()
    if (
        resolved != root
        and root not in resolved.parents
    ):
        raise ValueError(
            "Artifact path escapes extraction "
            f"root: {member!r}"
        )
    return resolved


total = 0
with zipfile.ZipFile(zip_path) as archive:
    for info in archive.infolist():
        mode = (
            info.external_attr >> 16
        ) & 0xFFFF
        if stat.S_ISLNK(mode):
            raise ValueError(
                "Symlink is not allowed in "
                f"artifact ZIP: {info.filename!r}"
            )

        total += info.file_size
        if total > MAX_ARTIFACT_BYTES:
            raise ValueError(
                "Artifact ZIP expands beyond "
                "the 512 MiB safety ceiling."
            )

        target = manifest_member_path(
            extract_root,
            info.filename,
        )

        if info.is_dir():
            target.mkdir(
                parents=True,
                exist_ok=True,
            )
            continue

        target.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        with (
            archive.open(info) as src,
            open(target, "wb") as dst,
        ):
            shutil.copyfileobj(
                src,
                dst,
            )

manifest_candidates = list(
    extract_root.rglob(
        "artifact-manifest.json"
    )
)
if len(manifest_candidates) != 1:
    raise ValueError(
        "Expected exactly one "
        "artifact-manifest.json."
    )

artifact_root = (
    manifest_candidates[0].parent
)
print("Artifact root:", artifact_root)

In [ ]:
manifest = json.loads(
    (
        artifact_root
        / "artifact-manifest.json"
    ).read_text(encoding="utf-8")
)

if (
    manifest.get("format")
    != "peft_adapter"
    or manifest.get("formatVersion")
    != 1
):
    raise ValueError(
        "Unsupported artifact format "
        "or version."
    )

if (
    not isinstance(
        manifest.get("files"),
        list,
    )
    or not manifest["files"]
):
    raise ValueError(
        "Artifact manifest has "
        "no file records."
    )

listed_paths = set()
calculated_total = 0

for record in manifest["files"]:
    rel = record.get("path")
    expected_hash = record.get("sha256")
    expected_bytes = record.get("bytes")

    if (
        not isinstance(rel, str)
        or rel in listed_paths
    ):
        raise ValueError(
            f"Invalid or duplicate "
            f"manifest path: {rel!r}"
        )

    if (
        not isinstance(
            expected_hash,
            str,
        )
        or len(expected_hash) != 64
    ):
        raise ValueError(
            f"Invalid SHA-256 entry "
            f"for {rel!r}."
        )

    path = manifest_member_path(
        artifact_root,
        rel,
    )

    if not path.is_file():
        raise ValueError(
            "Manifest-listed file "
            f"is missing: {rel}"
        )

    if path.stat().st_size != expected_bytes:
        raise ValueError(
            f"Size mismatch: {rel}"
        )

    if sha256_file(path) != expected_hash:
        raise ValueError(
            f"SHA-256 mismatch: {rel}"
        )

    listed_paths.add(rel)
    calculated_total += (
        path.stat().st_size
    )

actual_files = {
    path.relative_to(
        artifact_root
    ).as_posix()
    for path in artifact_root.rglob("*")
    if (
        path.is_file()
        and path.name
        != "artifact-manifest.json"
    )
}

if actual_files != listed_paths:
    extra = sorted(
        actual_files - listed_paths
    )
    missing = sorted(
        listed_paths - actual_files
    )
    raise ValueError(
        "Manifest/file-set mismatch. "
        f"Extra={extra}; missing={missing}"
    )

if (
    calculated_total
    != manifest.get("totalBytes")
):
    raise ValueError(
        "Manifest totalBytes does not "
        "match the listed files."
    )

required = {
    "adapter_model.safetensors",
    "adapter_config.json",
    "metrics.json",
    "provenance.json",
    "MODEL_CARD.md",
}
if not required.issubset(actual_files):
    raise ValueError(
        "Artifact is missing required files: "
        f"{sorted(required - actual_files)}"
    )

if not any(
    path.startswith("tokenizer/")
    for path in actual_files
):
    raise ValueError(
        "Artifact has no tokenizer directory."
    )

print(
    "✓ Manifest verified: "
    f"{len(listed_paths)} files, "
    f"{calculated_total / (1024**2):.2f} MiB"
)

## 3. Resolve immutable base-model provenance

The artifact may select only the exact model and commit recorded during training.
`trustRemoteCode` must be false and the revision must be a 40-character hexadecimal commit SHA.

In [ ]:
provenance = json.loads(
    (
        artifact_root
        / "provenance.json"
    ).read_text(encoding="utf-8")
)

if (
    provenance.get("artifactFormat")
    != "peft_adapter"
):
    raise ValueError(
        "provenance.json does not "
        "declare a PEFT adapter artifact."
    )

if (
    provenance.get(
        "artifactFormatVersion"
    )
    != 1
):
    raise ValueError(
        "Unsupported provenance artifact "
        "format version."
    )

if (
    provenance.get("trustRemoteCode")
    is not False
):
    raise ValueError(
        "Artifact provenance does not "
        "prove trustRemoteCode=false."
    )

base_model_id = provenance.get(
    "baseModel"
)
base_revision = provenance.get(
    "baseModelRevision"
)
expected_revision = provenance.get(
    "baseModelRevisionExpected"
)

if (
    not isinstance(base_model_id, str)
    or not base_model_id
):
    raise ValueError(
        "Missing baseModel provenance."
    )

if (
    not isinstance(base_revision, str)
    or not re.fullmatch(
        r"[0-9a-f]{40}",
        base_revision,
    )
):
    raise ValueError(
        "baseModelRevision is not an "
        "immutable 40-character commit SHA."
    )

if expected_revision != base_revision:
    raise ValueError(
        "Recorded loaded and expected "
        "base revisions disagree."
    )

print(
    "Model key:",
    provenance.get("modelKey"),
)
print("Base model:", base_model_id)
print("Pinned revision:", base_revision)
print(
    "Base license:",
    provenance.get("baseModelLicense"),
)

## 4. Load the pinned base and attach the adapter

`4bit` is the default inference mode for Colab GPU memory efficiency. It does not change which
base revision is loaded.

In [ ]:
INFERENCE_LOAD_MODE = "4bit"  # @param ["4bit", "full"]

from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

tokenizer = (
    AutoTokenizer.from_pretrained(
        artifact_root / "tokenizer"
    )
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def bf16_supported():
    return (
        torch.cuda.is_available()
        and torch.cuda.is_bf16_supported()
    )


quantization_config = None
if INFERENCE_LOAD_MODE == "4bit":
    quantization_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(
                torch.bfloat16
                if bf16_supported()
                else torch.float16
            ),
        )
    )

dtype = (
    torch.bfloat16
    if bf16_supported()
    else torch.float16
)

base = (
    AutoModelForCausalLM.from_pretrained(
        base_model_id,
        revision=base_revision,
        trust_remote_code=False,
        dtype=dtype,
        quantization_config=(
            quantization_config
        ),
        device_map={"": 0},
    )
)

loaded_revision = getattr(
    base.config,
    "_commit_hash",
    None,
)
if (
    loaded_revision
    and loaded_revision != base_revision
):
    raise RuntimeError(
        f"Loaded revision {loaded_revision} "
        "does not match artifact provenance "
        f"{base_revision}."
    )

model = PeftModel.from_pretrained(
    base,
    artifact_root,
)
model.eval()
print(
    "✓ Pinned base model + adapter loaded."
)

## 5. Generate from new prompts

In [ ]:
PROMPTS = [
    (
        "Ipaliwanag sa simpleng Filipino "
        "kung ano ang machine learning."
    ),
    (
        "Magbigay ng tatlong praktikal "
        "na paraan para mabawasan ang "
        "basura sa opisina."
    ),
]


def generate_reply(
    prompt,
    max_new_tokens=128,
):
    rendered = (
        tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    inputs = tokenizer(
        rendered,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=(
                tokenizer.pad_token_id
            ),
        )

    generated = output[
        0,
        inputs["input_ids"].shape[1] :,
    ]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


rows = [
    {
        "prompt": prompt,
        "response": generate_reply(prompt),
    }
    for prompt in PROMPTS
]
pd.DataFrame(rows)

If the notebook reaches this point, the portable artifact has passed:

- ZIP containment and symlink checks;
- manifest version/file-set/size/SHA-256 verification;
- immutable base revision validation;
- `trust_remote_code=False` enforcement;
- pinned base-model reload; and
- adapter-backed generation.

The training dataset was not needed.